# S12 – Coating, synthesis and row identity checked against the publications

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
from S12_comments import Audit, apply_comments   # the code of the comments of the step, apart from the data handling

INPUT = "../../S11_Unresolved_generic_iron_oxide_removed/S11.xlsx"
OUTPUT = "../S12.xlsx"
COMMENTS = "../S12_comments.xlsx"
AUDIT = "../S12_audit.xlsx"
STEP = "S12"

df = pd.read_excel(INPUT, keep_default_na=False, na_values=[""]).astype(object)
print(len(df), "records before the step")
audit = Audit(STEP, df, before=INPUT)
put, drop = audit.put, audit.drop                  # a rule sets a value / removes records through the audit, which records the change
changes, removed = audit.changes, audit.removed    # the ledger of the step

3435 records before the step


### 1. Coating labels: one spelling

In [2]:
COATING_LABELS = {   # label in the source file -> label
    '0': 'Uncoated',
    '0.0': 'Uncoated',
    '': 'Uncoated',
    'nan': 'Uncoated',
    'apigenin': 'Apigenin',
    'AscorbicAcid': 'Ascorbic acid',
    'Citrate and PVP': 'Citrate/PVP',
    'citrate': 'Citrate',
    'Gum Arabic': 'Gum arabic',
    'OleicAcid': 'Oleic acid',
    'Ph': 'Phosphonate',
    'pomegranate': 'Pomegranate',
    'tannic acid': 'Tannic acid',
    'Tween80': 'Tween 80',
}
for i in df.index:
    v = df.at[i, "Coat/Functional Group"]
    s = "" if pd.isna(v) else str(v).strip()
    if s in COATING_LABELS:
        put(i, "Coat/Functional Group", COATING_LABELS[s], f"{STEP}_coat_label",
            "coating label normalisation (0/blank -> Uncoated; spelling variants of the same chemistry)", outside=False)

### 2. Type: all retained materials are inorganic

In [3]:
for i in df.index:
    if df.at[i, "Type"] != "I":
        put(i, "Type", "I", f"{STEP}_type", "all retained materials are inorganic")

### 3. Values from the publication review: the comments of this step

In [4]:
# every value from the publication review: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S12_curated_L1_doi', 'S12_curated_L2_material', 'S12_curated_L3_coat', 'S12_curated_L4_diameter', 'S12_curated_L7_row'])

1316 comments applied (S12_curated_L1_doi, S12_curated_L2_material, S12_curated_L3_coat, S12_curated_L4_diameter, S12_curated_L7_row): 1316 values set, 0 records removed


### 4. Corrections from the check against the publications: the comments of this step

In [5]:
# every correction from the check of this field against the publications: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S12_publication_check_01', 'S12_publication_check_02', 'S12_publication_check_03', 'S12_publication_check_04'])

86 comments applied (S12_publication_check_01, S12_publication_check_02, S12_publication_check_03, S12_publication_check_04): 86 values set, 0 records removed


### 5. Synthesis categories: their final names

In [6]:
SYNTHESIS_NAMES = {   # category -> (final name, basis)
    'Commercial': ('Not synthesised by the authors',
                    'the category covers every material the authors did not synthesise themselves - purchased, or supplied by another '
                    'laboratory or company'),
}
for i in df.index:
    v = df.at[i, "Synthesis_Method"]
    if not pd.isna(v) and str(v) in SYNTHESIS_NAMES:
        to, basis = SYNTHESIS_NAMES[str(v)]
        put(i, "Synthesis_Method", to, f"{STEP}_synthesis_vocabulary", "final name of the synthesis category", basis, outside=False)

### The table after the step and its audit

In [7]:
# the table after the step; the audit module compares it with the table before the step and writes the audit workbook
df.to_excel(OUTPUT, index=False)
audit.write(AUDIT)

0 records removed and 5315 cells changed compared with the table before the step


3435 records after the step | 0 records removed | 6232 cells changed | ../S12_audit.xlsx
